# Yapay Zeka Dersi - Ödev 2

In [ ]:
import pandas as pd
import os
import numpy as np
from gensim.models import Word2Vec
from sklearn.metrics.pairwise import cosine_similarity
import seaborn as sns
import matplotlib.pyplot as plt

## Görev 1: Word2Vec Vektörleştirme

In [ ]:
def load_data():
    lemmatized_df = pd.read_csv("Lemmatized_Veri_Seti.csv")
    stemmed_df = pd.read_csv("Stemmed_Veri_Seti.csv")
    lemmatized_texts = lemmatized_df["Lemmatized_Metin"].fillna("").apply(lambda x: x.split()).tolist()
    stemmed_texts = stemmed_df["Stemmed_Metin"].fillna("").apply(lambda x: x.split()).tolist()
    return lemmatized_df, lemmatized_texts, stemmed_df, stemmed_texts

In [ ]:
def train_and_save_models(lemmatized_texts, stemmed_texts):
    if not os.path.exists("model"):
        os.makedirs("model")
    
    parameters = [ 
        {"model_type": "cbow", "window": 2, "vector_size": 100},
        {"model_type": "skipgram", "window": 2, "vector_size": 100},
        {"model_type": "cbow", "window": 4, "vector_size": 100},
        {"model_type": "skipgram", "window": 4, "vector_size": 100},
        {"model_type": "cbow", "window": 2, "vector_size": 300},
        {"model_type": "skipgram", "window": 2, "vector_size": 300},
        {"model_type": "cbow", "window": 4, "vector_size": 300},
        {"model_type": "skipgram", "window": 4, "vector_size": 300}
    ]
    
    models = {}
    for param in parameters:
        sg = 1 if param["model_type"] == "skipgram" else 0
        window = param["window"]
        vector_size = param["vector_size"]
        
        # Lemmatized
        model_name_lem = f"lemmatized_model_{param['model_type']}_window{window}_dim{vector_size}"
        model_lem = Word2Vec(sentences=lemmatized_texts, vector_size=vector_size, window=window, sg=sg, min_count=1, workers=4)
        model_lem.save(f"model/{model_name_lem}.model")
        models[model_name_lem] = model_lem
        
        # Stemmed
        model_name_stem = f"stemmed_model_{param['model_type']}_window{window}_dim{vector_size}"
        model_stem = Word2Vec(sentences=stemmed_texts, vector_size=vector_size, window=window, sg=sg, min_count=1, workers=4)
        model_stem.save(f"model/{model_name_stem}.model")
        models[model_name_stem] = model_stem
        
    return models

In [ ]:
lem_df, lem_texts, stem_df, stem_texts = load_data()
models = train_and_save_models(lem_texts, stem_texts)
print("Görev 1: Veriler yüklendi ve modeller eğitildi.")

## Görev 2: Metin Benzerliklerinin Hesaplanması

In [ ]:
def get_sentence_vector(sentence_tokens, model, vector_size):
    vectors = [model.wv[word] for word in sentence_tokens if word in model.wv]
    if len(vectors) == 0:
        return np.zeros(vector_size)
    return np.mean(vectors, axis=0)

def calculate_similarity(input_text, models, df, texts_list):
    input_tokens = input_text.split()
    results = {}
    for model_name, model in models.items():
        v_size = 100 if "dim100" in model_name else 300
        input_vector = get_sentence_vector(input_tokens, model, v_size)
        
        doc_vectors = []
        for doc_tokens in texts_list:
            doc_vectors.append(get_sentence_vector(doc_tokens, model, v_size))
            
        similarities = cosine_similarity(input_vector.reshape(1, -1), np.array(doc_vectors))[0]
        top_indices = similarities.argsort()[-6:][::-1]
        
        top_5_docs = []
        for idx in top_indices:
            top_5_docs.append({
                "index": idx,
                "tarif_adi": df.iloc[idx]["Tarif_Adi"],
                "score": similarities[idx]
            })
            if len(top_5_docs) == 5: break
        results[model_name] = top_5_docs
    return results

In [ ]:
sample_text_lem = lem_df.iloc[0]["Lemmatized_Metin"] 
sample_text_stem = stem_df.iloc[0]["Stemmed_Metin"]

lem_models = {k: v for k, v in models.items() if "lemmatized" in k}
lem_results = calculate_similarity(sample_text_lem, lem_models, lem_df, lem_texts)

stem_models = {k: v for k, v in models.items() if "stemmed" in k}
stem_results = calculate_similarity(sample_text_stem, stem_models, stem_df, stem_texts)

all_results = {**lem_results, **stem_results}
print("Görev 2: Benzerlikler hesaplandı.")

## Görev 3: Değerlendirme ve Jaccard Heatmap

In [ ]:
def jaccard_similarity(list1, list2):
    set1, set2 = set(list1), set(list2)
    return len(set1.intersection(set2)) / len(set1.union(set2))

In [ ]:
model_names = list(all_results.keys())
n = len(model_names)
jaccard_matrix = np.zeros((n, n))

for i in range(n):
    for j in range(n):
        ids_i = [item["index"] for item in all_results[model_names[i]]]
        ids_j = [item["index"] for item in all_results[model_names[j]]]
        jaccard_matrix[i, j] = jaccard_similarity(ids_i, ids_j)

plt.figure(figsize=(14, 10))
sns.heatmap(jaccard_matrix, annot=True, cmap="YlGnBu", xticklabels=model_names, yticklabels=model_names, fmt=".2f")
plt.title("Jaccard Benzerliği Heatmap (İlk 5 Sonuç)")
plt.tight_layout()
plt.show()